In [0]:
!pip install -U pypdf

In [0]:

!pip install -U langchain-text-splitters
!pip install -U databricks_langchain
!pip install pandas
!pip install faiss-cpu

In [0]:
dbutils.library.restartPython()

In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pyspark.sql import functions as F


In [0]:
vol_landing_path="/Volumes/rag_on_databricks/landing/vol_landing/Orion_Docs/"
dbutils.fs.ls(vol_landing_path)

In [0]:
from pyspark.sql.functions import expr

# Read all files from the documents volume
docs_df = spark.read.format("binaryFile").load(vol_landing_path)

# Parse each document using ai_parse_document (use expr to call the SQL AI function)
parsed_df = docs_df.withColumn("parsed_content", 
                           expr(f"""ai_parse_document(content, map(
                                "version", "2.0",
                                "imageOutputPath", "{vol_landing_path}parsed_images/"
                               ))""")
                          )
# Drop binary content
parsed_df = parsed_df.drop("content")

# Display a sample of the parsed results
display(parsed_df)


In [0]:

pages=[]

for file in dbutils.fs.ls(vol_landing_path):
    reader=PdfReader(file.path.replace("dbfs:", ""))
    for page_num, page in enumerate(reader.pages,start=1):
        text= page.extract_text()
        pages.append(
            {
                "text":text,
                "page_num":page_num
            }
        )
print(len(pages)) #1201
    

In [0]:
spark.sql(f"LIST '{vol_landing_path}parsed_images/'").display()

In [0]:
# Save the parsed results as a Delta table for easy querying and sharing
catalog="rag_on_databricks"
schema="landing"

output_table = f"{catalog}.{schema}.docs_parsed"

# Overwrite the table if it already exists
parsed_df.write.format("delta").mode("overwrite").saveAsTable(output_table)

print(f"✅ Parsed results saved to Delta table: {output_table}")

In [0]:
%sql
select * from rag_on_databricks.landing.docs_parsed

In [0]:
catalog="rag_on_databricks"
schema="landing"

parsed_table = f"{catalog}.{schema}.docs_parsed"
chunked_table = f"{catalog}.{schema}.docs_chunked"

parsed_df = spark.read.table(parsed_table)

print(f"Loaded parsed documents from: {parsed_table}")
parsed_df.printSchema()

In [0]:
from pyspark.sql.functions import expr

# Choose a Databricks foundation model (or your own serving endpoint name)
ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

# Example prompt for the LLM
prompt_prefix = '''
You are a helpful assistant. Given a JSON object representing a parsed document (with pages, elements, and metadata), convert the content into clean, readable markdown. Use "== page ==" to separate each page. Preserve important structure such as headers, tables, and captions. Do not include any JSON or code blocks in the output—just the clean markdown text.

JSON:

'''

# Apply ai_query to batch process the parsed JSON text
# Note: Claude models do not support responseFormat type "text"; omit it for plain-text output.
transformed_df = (
    parsed_df.withColumn(
        "clean_markdown_text",
        expr(f"""
          ai_query(
            '{ENDPOINT}',
            CONCAT('{prompt_prefix}', CAST(parsed_content AS STRING))
          )
        """)
    )
)

display(transformed_df.select("path", "clean_markdown_text"))

In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Set chunking parameters
CHUNK_SIZE = 2000
CHUNK_OVERLAP = 200

# Build the text splitter (similar to Cell 16)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n== page ==\n", "== page ==", "\n\n", "\n", " ", ""]
)

# Collect the cleaned documents (similar to Cell 17 approach)
docs_list = transformed_df.select("path", "clean_markdown_text").collect()

# Split each document into chunks
chunks = []
for doc in docs_list:
    path = doc["path"]
    text = doc["clean_markdown_text"]
    
    if text and text.strip():
        # Split the text into chunks
        chunks_subset = splitter.split_text(text)
        
        # Create chunk records with document path only
        for chunk in chunks_subset:
            if chunk.strip():
                chunks.append({
                    "path": path,
                    "chunk": chunk
                })

print(f"Total chunks created: {len(chunks)}")

# Convert back to Spark DataFrame
df_chunks = spark.createDataFrame(chunks)

# Display the resulting chunked DataFrame
display(df_chunks)

In [0]:

# Add a unique, incremental id column before saving
df_chunks = df_chunks.withColumn("id", F.monotonically_increasing_id())

# Save the chunked data with id to the Delta table for retrieval and embedding
df_chunks.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(chunked_table)

display(spark.read.table(chunked_table))

In [0]:
chunked_table

In [0]:
from databricks_langchain import DatabricksEmbeddings

embedding_model = DatabricksEmbeddings(endpoint="databricks-bge-large-en")

In [0]:
# Load chunks from Delta table
df_chunks = spark.table(chunked_table).toPandas()
chunk_list = df_chunks["chunk"].tolist()

print(f"Loaded {len(chunk_list)} chunks from {chunked_table}")

In [0]:
# Generate embeddings in batches (same approach as Cell 25)
import time

batch_size = 15
all_embeddings = []

print(f"Processing {len(chunk_list)} chunks (batch_size={batch_size})\n")
start_time = time.time()

for i in range(0, len(chunk_list), batch_size):
    batch = chunk_list[i:i + batch_size]
    batch_embeddings = embedding_model.embed_documents(batch)
    all_embeddings.extend(batch_embeddings)
    print(f"Processed {min(i + batch_size, len(chunk_list))}/{len(chunk_list)} chunks")
    time.sleep(2)

embeddings = all_embeddings
elapsed = time.time() - start_time
print(f"\n✅ Generated {len(embeddings)} embeddings in {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"Dimensions: {len(embeddings[0])}")

In [0]:
# Add embeddings to dataframe and save to Delta table
df_chunks['embedding'] = embeddings

# Define output table
embeddings_table = f"{catalog}.{schema}.docs_with_embeddings"

# Convert to Spark and save
spark_df = spark.createDataFrame(df_chunks)
spark_df.write.format("delta").mode("overwrite").saveAsTable(embeddings_table)

print(f"\n✅ Saved {len(df_chunks)} chunks with embeddings to: {embeddings_table}")
display(spark.table(embeddings_table).limit(5))

In [0]:
import faiss
import numpy as np

# Load embeddings from Delta table
df_embedded = spark.table(embeddings_table).toPandas()

# Prepare data for FAISS
chunks_data = df_embedded[['id', 'path', 'chunk']].to_dict('records')
embeddings_array = np.array(df_embedded['embedding'].tolist(), dtype='float32')

print(f"Loaded {len(embeddings_array)} embeddings")
print(f"Embedding dimensions: {embeddings_array.shape[1]}")

# Normalize embeddings for cosine similarity (Inner Product)
norms = np.linalg.norm(embeddings_array, axis=1, keepdims=True)
norms[norms == 0] = 1e-12
normalized_embeddings = embeddings_array / norms

print("\n🔨 Building FAISS index...")

# Build FAISS index with Inner Product (cosine similarity for normalized vectors)
dimension = embeddings_array.shape[1]
nlist = max(1, len(embeddings_array) // 50)  # Adaptive cluster size

quantizer = faiss.IndexFlatIP(dimension)  # Inner Product = Cosine Similarity
index = faiss.IndexIVFFlat(quantizer, dimension, nlist, faiss.METRIC_INNER_PRODUCT)

# Train and add vectors
index.train(normalized_embeddings)
index.add(normalized_embeddings)

print(f"\n✅ FAISS index built successfully!")
print(f"   - Total vectors: {index.ntotal}")
print(f"   - Clusters (nlist): {nlist}")
print(f"   - Metric: Inner Product (Cosine Similarity)")
print(f"   - Index type: IVFFlat")

In [0]:
# Optional: Save FAISS index to Volume for reuse
import pickle

index_save_path = "/Volumes/rag_on_databricks/landing/vol_landing/faiss_index/"

# Create directory if it doesn't exist
dbutils.fs.mkdirs(index_save_path)

# Save FAISS index
faiss.write_index(index, index_save_path.replace("dbfs:", "") + "index.faiss")

# Save chunks metadata
with open(index_save_path.replace("dbfs:", "") + "chunks_data.pkl", "wb") as f:
    pickle.dump(chunks_data, f)

print(f"\n✅ FAISS index and metadata saved to: {index_save_path}")
print("\nTo load later:")
print(f"  index = faiss.read_index('{index_save_path}index.faiss')")
print(f"  with open('{index_save_path}chunks_data.pkl', 'rb') as f:")
print(f"      chunks_data = pickle.load(f)")

In [0]:
# Load the saved FAISS index and metadata from Volume
import faiss
import pickle
import numpy as np

index_save_path = "/Volumes/rag_on_databricks/landing/vol_landing/faiss_index/"

print("Loading FAISS index from Volume...")

# Load FAISS index
index = faiss.read_index(index_save_path.replace("dbfs:", "") + "index.faiss")

# Load chunks metadata
with open(index_save_path.replace("dbfs:", "") + "chunks_data.pkl", "rb") as f:
    chunks_data = pickle.load(f)

print(f"✅ Loaded FAISS index with {index.ntotal} vectors")
print(f"✅ Loaded metadata for {len(chunks_data)} chunks")
print(f"\n📌 Ready to use retrieve() function with saved index!")

In [0]:
def retrieve(query, k=4):
    """
    Retrieve top-k most relevant chunks for a query using FAISS.
    
    Args:
        query (str): User's question
        k (int): Number of chunks to retrieve
    
    Returns:
        list: Top-k chunks with similarity scores
    """
    # Embed and normalize query
    query_vector = np.array(embedding_model.embed_query(query), dtype='float32').reshape(1, -1)
    
    # Normalize for cosine similarity
    query_norm = np.linalg.norm(query_vector, axis=1, keepdims=True)
    query_norm[query_norm == 0] = 1e-12
    normalized_query = query_vector / query_norm
    
    # Search FAISS index
    scores, indices = index.search(normalized_query, k)
    
    # Return results with metadata
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "chunk": chunks_data[idx]['chunk'],
            "path": chunks_data[idx]['path'],
            "id": chunks_data[idx]['id'],
            "score": float(scores[0][i])
        })
    
    return results

# Test the retrieval
test_query = "How does the Orion system prevent overheating?"
test_results = retrieve(test_query, k=3)

print(f"\n🔍 Query: {test_query}")
print(f"\n✅ Retrieved {len(test_results)} chunks:\n")
for i, result in enumerate(test_results, 1):
    print(f"{i}. Score: {result['score']:.4f}")
    print(f"   Path: {result['path'].split('/')[-1]}")
    print(f"   Chunk preview: {result['chunk'][:150]}...\n")

In [0]:
def create_rag_prompt(retrieved_chunks, question):
    """
    Create augmented prompt with retrieved context for LLM.
    """
    context = "\n\n".join([
        f"Source {i+1} (Score: {chunk['score']:.3f})\n{chunk['chunk']}"
        for i, chunk in enumerate(retrieved_chunks)
    ])
    
    prompt = f"""You are an expert AI assistant.

Use ONLY the information provided in the context below to answer the user's question.

Rules:
- Answer only from the provided context.
- Do not make up information.
- If the answer is not available in the context, reply: "I couldn't find the answer in the provided documents."
- Keep the answer clear, concise, and professional.
- Use bullet points whenever appropriate.

Context:
{context}

Question:
{question}

Answer:"""
    
    return prompt

def RAG(query, k=4):
    """
    Complete RAG pipeline: Retrieve + Augment + Generate.
    
    Args:
        query (str): User's question
        k (int): Number of chunks to retrieve
    
    Returns:
        dict: Question, answer, and sources
    """
    # 1. Retrieve relevant chunks
    retrieved_chunks = retrieve(query, k=k)
    
    # 2. Create augmented prompt
    prompt = create_rag_prompt(retrieved_chunks, query)
    
    # 3. Return prompt and sources (ready for LLM)
    return {
        "question": query,
        "prompt": prompt,
        "sources": retrieved_chunks
    }

# Example usage
user_query = "What are the safety features of the Orion A1?"
rag_result = RAG(user_query, k=4)

print(f"\n💬 Query: {rag_result['question']}")
print(f"\n📚 Retrieved {len(rag_result['sources'])} sources")
print(f"\n📤 Augmented Prompt ready for LLM:")
print("=" * 80)
print(rag_result['prompt'][:500] + "...")
print("=" * 80)
print("\n✅ Ready to send to LLM (Cell 37-39)!")

In [0]:
# Display summary of parsed documents
import json

# Show document-level metadata
print("=== Parsed Documents Summary ===")
for row in parsed_df.collect():
    path = row['path'].split('/')[-1]
    parsed = json.loads(row['parsed_content'])
    
    num_pages = len(parsed.get('document', {}).get('pages', []))
    num_elements = len(parsed.get('document', {}).get('elements', []))
    
    print(f"\n📄 {path}")
    print(f"   Pages: {num_pages}")
    print(f"   Elements extracted: {num_elements}")
    
    # Count element types
    element_types = {}
    for elem in parsed.get('document', {}).get('elements', []):
        elem_type = elem.get('type', 'unknown')
        element_types[elem_type] = element_types.get(elem_type, 0) + 1
    
    print(f"   Element breakdown: {element_types}")

In [0]:
%skip
from langchain_community.document_loaders import PyPDFLoader

# Initialize the loader with the path to your PDF file
loader = PyPDFLoader("path/to/your/document.pdf")

# Load the pages (each page becomes a LangChain Document object)
pages = loader.load()

# You can access the content and metadata of a specific page
print(pages[0].page_content)  # The text content of the page
print(pages[0].metadata)      # Metadata (e.g., source file path, page number)


In [0]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1500,chunk_overlap=300, separators=["\n\n", "\n", " ", ", ",""])
splitter.split_text(pages[0]["text"])

In [0]:
chunks=[]
for i, page in enumerate(pages):
    text=page["text"]
    chunks_subset=splitter.split_text(text)
    for j,chunk in enumerate(chunks_subset):
        chunks.append({
            "chunk":chunk,
            "id": f'chunk_id_{page["page_num"]}_{j}'
        })

print(len(chunks))

In [0]:
#chunks[100]

In [0]:
np.array(embedding_model.embed_query("What is the meaning of life ?")).shape

In [0]:
# Generate embeddings for all chunks
import time

chunk_list = data["chunk"].tolist()
batch_size = 50  # Reasonable batch size for Databricks
all_embeddings = []

print(f"Processing {len(chunk_list)} chunks (batch_size={batch_size})\n")
start_time = time.time()

for i in range(0, len(chunk_list), batch_size):
    batch = chunk_list[i:i + batch_size]
    batch_embeddings = embedding_model.embed_documents(batch)
    all_embeddings.extend(batch_embeddings)
    print(f"Processed {min(i + batch_size, len(chunk_list))}/{len(chunk_list)} chunks")
    time.sleep(2)  # 2 second delay between batches to avoid rate limits

embeddings = all_embeddings
elapsed = time.time() - start_time
print(f"\nGenerated {len(embeddings)} embeddings in {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"Dimensions: {len(embeddings[0])}")

In [0]:
np.array(embeddings).shape #embedding to convert that into dataframe and that(data) df is added in delta table..


In [0]:
def normalize(vector):
    norms = np.linalg.norm(vector, axis=1, keepdims=True)
    norms[norms==0] = 1e-12
    normalized_embds = vector / norms  # Make them unit vectors
    return normalized_embds

In [0]:
# Normalize embeddings, then save to Delta table
normalized_embeddings = normalize(np.array(embeddings))

spark_df = spark.createDataFrame([
    (
        chunks[i]['id'],
        chunks[i]['chunk'],
        normalized_embeddings[i].tolist()  #Store normalized embeddings
    )
    for i in range(len(chunks))
], ["id", "text", "embedding"])

# Write to Delta Lake
spark_df.write.format("delta").mode("overwrite").saveAsTable("rag_on_databricks.landing.document_embeddings")

print(f"Saved {len(chunks)} chunks to Delta table")

In [0]:
%sql
select * from rag_on_databricks.landing.document_embeddings

In [0]:
#Build Faiss index for fast search
import faiss

# Load normalized embeddings from Delta table
df = spark.table("rag_on_databricks.landing.document_embeddings").toPandas()
chunks_loaded = df[['id', 'text']].rename(columns={'text': 'chunk'}).to_dict('records')
normalized_chunk = np.array(df['embedding'].tolist())

print(f"Loaded {len(normalized_chunk)} normalized embeddings from Delta table")

dimension = 1024
nlist = 1 #50 #Cluster/group size

quantizer = faiss.IndexFlatIP(dimension)  # Inner product = Cosine Similarity for normalized
index = faiss.IndexIVFFlat(quantizer, dimension, nlist, faiss.METRIC_INNER_PRODUCT)

index.train(normalized_chunk) #how to divide these vectors into 50 cluster
index.add(normalized_chunk) #all your embeddings into those clusters

print(f"Faiss index built with {index.ntotal} vectors")

# Option B: Exact search (simpler, good for <100k vectors)
# index = faiss.IndexFlatIP(dimension)
# index.add(normalized_chunk)


In [0]:
normalized_chunk.shape

In [0]:
# Cell 18: Fast retrieval with Faiss
def retrieve(query, k=4):
    # Embed and normalize query
    query_vector = np.array(embedding_model.embed_query(query)).reshape(1, -1)
    normalized_query = normalize(query_vector)
    
    # Search Faiss index
    scores, indices = index.search(normalized_query, k)
    
    # Return results (using chunks_loaded from Delta table)
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "chunk": chunks_loaded[idx]['chunk'],
            "id": chunks_loaded[idx]['id'],
            "score": float(scores[0][i])
        })
    
    return results

In [0]:
from databricks_langchain import ChatDatabricks

model = ChatDatabricks(
    endpoint="databricks-meta-llama-3-1-8b-instruct",
    max_tokens=500,
    temperature=0.1
)

In [0]:
def create_prompt(retrieved_sources, question):
    context = "\n\n".join(
        [
            f"Source {i+1}\n{doc['chunk']}"
            for i, doc in enumerate(retrieved_sources)
        ]
    )

    prompt = f"""
You are an expert AI assistant.

Use ONLY the information provided in the context below to answer the user's question.

Rules:
- Answer only from the provided context.
- Do not make up information.
- If the answer is not available in the context, reply:
  "I couldn't find the answer in the provided documents."
- Keep the answer clear, concise, and professional.
- Use bullet points whenever appropriate.

Context:
{context}

Question:
{question}

Answer:
"""

    return prompt

In [0]:

def RAG(query):
    retrieved_sources = retrieve(query, k=4)

    prompt = create_prompt(retrieved_sources, query)

    response = model.invoke(prompt)

    answer=response.content
    return {
        "question": query,
        "answer": answer,
        "sources": retrieved_sources
    }

user_query = "DCP Architecture ?"#"Detailed DCP Architecture: Inbound & Outbound Data Flow ?" 
#"what is the big data in the hadoop ecosystem ?" 
# "What is the YARN Architecture ?"
# "what is the big data in the hadoop ecosystem ?" 
# "what is the capital of france ?"
# "What is sql language related with hadoop and data engineering context ?"
# "What is hdfs in hadoop ?"
# "What is mapreduce in the hadoop ecosystem ?"

result=RAG(user_query)


In [0]:
#check on PII docs

In [0]:
print(result["answer"])